# Setup

In [4]:
# Заставляем ноутбук обновлять импорты автоматически (если ты изменил код в src/)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import sys
from pathlib import Path
from hydra.utils import instantiate
import matplotlib.pyplot as plt

# 1. Находим корень проекта (поднимаемся на уровень вверх из папки notebooks)
# Если ты в notebooks/, то родительская папка — это корень проекта
project_root = Path.cwd().parent

# 2. Добавляем в sys.path, если его там еще нет
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added {project_root} to sys.path")

# 3. Теперь импорт сработает
from src.utils.notebook_setup import init_nlp_notebook
cfg = init_nlp_notebook()

# Дальше работаешь с конфигом:
print(f"Config loaded. Model: {cfg.model.architecture.model_name}")

NLP Environment ready. Config loaded: main
Config loaded. Model: DeepPavlov/rubert-base-cased


# Data Loading

In [8]:
# Загружаем датасет, используя Hydra
dataset = instantiate(cfg.data.dataset_loader) 
# Пример вывода структуры
print(f"Dataset schema: {dataset['train'].column_names}")

ConfigAttributeError: Key 'dataset_loader' is not in struct
    full_key: data.dataset_loader
    object_type=dict

# Tokenization & Sequence Length Analysis

In [ ]:
# Пример анализа длин
lengths = [len(tokenizer.encode(x['text'])) for x in dataset['train'].select(range(min(1000, len(dataset['train']))))]

plt.hist(lengths, bins=50)
plt.title("Distribution of Token Lengths")
plt.show()

# Artifact & Noise Identification

In [ ]:
import pandas as pd
import re

# 1. Определяем паттерны "шума" (можно расширять под свои данные)
noise_patterns = {
    "extra_whitespace": r"\s{2,}",
    "html_tags": r"<[^>]+>",
    "non_printable": r"[^\x20-\x7E\u0400-\u04FF\n]", # Только ASCII + Кириллица
    "truncated_lines": r"^\s*\.\.\.\s*$",
}

def analyze_noise(text_list):
    stats = {}
    for name, pattern in noise_patterns.items():
        count = sum(1 for text in text_list if re.search(pattern, text))
        stats[name] = count / len(text_list)
    return stats

# Применяем на выборке
sample_data = [x['text'] for x in dataset['train'].select(range(min(5000, len(dataset['train']))))]
noise_report = analyze_noise(sample_data)

# Выводим отчет в виде таблицы
df_noise = pd.DataFrame.from_dict(noise_report, orient='index', columns=['percentage'])
print("--- Noise Artifacts Report ---")
print(df_noise)

# Если процент высокий — нужно править препроцессинг до обучения

# Quality Audit

In [ ]:
def check_round_trip(text, tokenizer):
    tokens = tokenizer.encode(text)
    decoded = tokenizer.decode(tokens, skip_special_tokens=False)
    return tokens, decoded

# Берем сложные примеры (длинные или с символами)
sample_texts = [x['text'] for x in dataset['train'].select(range(5))]

print(f"{'Original':<30} | {'Round-Trip (Decoded)'}")
print("-" * 80)

for text in sample_texts:
    tokens, decoded = check_round_trip(text, tokenizer)
    # Сравниваем, не потерялось ли чего важного
    # (могут остаться служебные токены, это ок, главное — текст)
    print(f"{text[:25]+'...':<30} | {decoded[:50]+'...'}")

# ПРОВЕРКА СПЕЦ-ТОКЕНОВ
# Убедись, что твои маркеры начала/конца (например <|start_header_id|>) 
# не превращаются в [UNK] или разбиваются на части.
test_marker = "<|start_header_id|>"
print(f"\nMarker '{test_marker}' encoding: {tokenizer.encode(test_marker)}")